In [0]:
%sql
-- Task 8: Why has the margin moved?
WITH by_line AS (
    SELECT
        CASE WHEN o.order_date <= DATE '2024-06-30' THEN 'FY2024' ELSE 'FY2025' END AS fin_year,
        p.product_line,
        SUM(l.line_revenue) AS revenue,
        SUM(l.cogs)         AS cogs
    FROM sizwe_sales_orders o
    JOIN sizwe_sales_lines  l ON l.order_id = o.order_id
    JOIN sizwe_products     p ON p.sku      = l.sku
    GROUP BY 1, 2
)
SELECT
    product_line,
    fin_year,
    ROUND(revenue, 0)                                                         AS revenue,
    ROUND(100.0 * (revenue - cogs) / revenue, 1)                              AS gross_margin_pct,
    ROUND(100.0 * revenue / SUM(revenue) OVER (PARTITION BY fin_year), 1)     AS share_of_revenue_pct
FROM by_line
ORDER BY product_line, fin_year;
-- Answer: The decline is a mix effect, not a pricing or cost problem. Every line's own margin is flat (Bulk Commercial 19.9% in both years, House Blend 38.3%, Decaf 35.8%, Single Origin 44.2% -> 44.3%), but Bulk Commercial, the lowest-margin line, grew from R110.3m to R161.5m (+46%) and its share of revenue jumped from 25.0% to 33.6%. The higher-margin lines lost share (House Blend 32.1% -> 28.7%, Single Origin 19.5% -> 17.6%), so the blended margin was dragged from 34.7% to 33.0%.

product_line,fin_year,revenue,gross_margin_pct,share_of_revenue_pct
Bulk Commercial,FY2024,1.10338581E8,19.9,25.0
Bulk Commercial,FY2025,1.6150025E8,19.9,33.6
Capsules,FY2024,5.682235E7,39.0,12.9
Capsules,FY2025,5.2052727E7,38.8,10.8
Decaf,FY2024,4.6300477E7,35.8,10.5
Decaf,FY2025,4.4342068E7,35.8,9.2
House Blend,FY2024,1.41746471E8,38.3,32.1
House Blend,FY2025,1.37784477E8,38.3,28.7
Single Origin,FY2024,8.5792118E7,44.2,19.5
Single Origin,FY2025,8.4320502E7,44.3,17.6
